In [28]:
import re
import openpyxl
from pptx import Presentation
from openpyxl.utils import column_index_from_string

def log_warning(message):
    print(f"WARNING: {message}")

def log_info(message):
    print(f"INFO: {message}")

# File paths
pptx_file = "template.pptx"
excel_file = "SCFT MODEL REPORT-Updated.xlsx"
output_file = "output.pptx"

# Load Excel file
log_info(f"Loading Excel file: {excel_file}")
try:
    workbook = openpyxl.load_workbook(excel_file, data_only=True)
    log_info(f"Loaded Excel. Sheets: {workbook.sheetnames}")
except Exception as e:
    log_warning(f"Failed to load Excel file: {e}")
    raise

# Load PowerPoint file
log_info(f"Loading PowerPoint file: {pptx_file}")
try:
    presentation = Presentation(pptx_file)
    log_info("PowerPoint file loaded")
except Exception as e:
    log_warning(f"Failed to load PowerPoint file: {e}")
    raise

INFO: Loading Excel file: SCFT MODEL REPORT-Updated.xlsx
INFO: Loaded Excel. Sheets: ['SCFT MODEL REPORT', 'Drive route ', 'Serving PCI Plot', 'RSRP Plot', 'RSRQ Plot', 'SINR Plot', 'DL Throughputplot', 'Rank Indicator Plot', 'Modulation Plot', 'Prediction Plot-RSRP', 'Prediction Plot-RSR', 'Prediction Plot-SINR', 'Prediction Plot-DL Throughput']
INFO: Loading PowerPoint file: template.pptx
INFO: PowerPoint file loaded


In [29]:
pattern = r"\{([^}]+)\}"
excluded_slides = [1]  # Hardcoded list of slides to exclude

for slide_index, slide in enumerate(presentation.slides, 1):
    if slide_index in excluded_slides:
        log_info(f"Skipping slide {slide_index} as it is in excluded list")
        continue
    log_info(f"Processing slide {slide_index}")
    placeholder_found = False
    for shape in slide.shapes:
        if shape.has_text_frame and not shape.has_table:
            for paragraph in shape.text_frame.paragraphs:
                text = paragraph.text
                matches = re.findall(pattern, text)
                for match in matches:
                    placeholder_found = True
                    print(f"Slide {slide_index}: Textbox paragraph placeholder found: {{{match}}}")
        elif shape.has_table:
            for row in shape.table.rows:
                for cell in row.cells:
                    full_text = cell.text
                    matches = re.findall(pattern, full_text)
                    if matches:
                        placeholder_found = True
                        # Cell contains at least one placeholder; keep only the placeholders
                        # log_info(f"Slide {slide_index}: Found cell with placeholders: '{full_text}'")
                        placeholders = [f"{{{match}}}" for match in matches]
                        # for placeholder in placeholders:
                        #     print(f"Slide {slide_index}: Table cell placeholder found: {placeholder}")
                        # Replace cell content with only the placeholders
                        new_text = " ".join(placeholders)
                        cell.text = new_text
                        log_info(f"Slide {slide_index}: Replaced cell content from '{full_text}' to '{new_text}'")
    if not placeholder_found:
        log_info(f"Slide {slide_index}: No placeholders found in table cells or textboxes")

INFO: Skipping slide 1 as it is in excluded list
INFO: Processing slide 2
INFO: Slide 2: Replaced cell content from 'RP-00553-MAYYIL
{SCFT MODEL REPORT :C2}' to '{SCFT MODEL REPORT :C2}'
INFO: Slide 2: Replaced cell content from '
{SCFT MODEL REPORT :D2}
RP-00553' to '{SCFT MODEL REPORT :D2}'
INFO: Slide 2: Replaced cell content from 'KNR
{SCFT MODEL REPORT :A2}' to '{SCFT MODEL REPORT :A2}'
INFO: Slide 2: Replaced cell content from 'KANNUR
{SCFT MODEL REPORT :B2}' to '{SCFT MODEL REPORT :B2}'
INFO: Slide 2: Replaced cell content from '674886- 0/1/2
{SCFT MODEL REPORT:
AI1}
{SCFT MODEL REPORT :AI2}
{SCFT MODEL REPORT :AI3}' to '{SCFT MODEL REPORT:
AI1} {SCFT MODEL REPORT :AI2} {SCFT MODEL REPORT :AI3}'
INFO: Slide 2: Replaced cell content from '03-01-2025
{SCFT MODEL REPORT:AK2}' to '{SCFT MODEL REPORT:AK2}'
INFO: Slide 2: Replaced cell content from 'Indoor
{SCFT MODEL REPORT :G2}' to '{SCFT MODEL REPORT :G2}'
INFO: Slide 2: Replaced cell content from 'GBT
{SCFT MODEL REPORT :Q2}' to '

In [31]:
presentation.save("refined_template.pptx")
log_info("Presentation saved successfully to 'refined_template.pptx'")

INFO: Presentation saved successfully to 'refined_template.pptx'
